In [27]:
import os
import re

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [31]:
def load_all_pdfs():
    folder_path = "data/raw"

    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):

        if filename.lower().endswith(".pdf"):

            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            docs = loader.load()

            for doc in docs:
                doc.metadata["document_name"] = filename

            all_docs.extend(docs)
            num_docs += 1

            print(f"Loaded: {filename} → {len(docs)} pages")

    print("\nTotal PDFs:", num_docs)
    print("Total pages:", len(all_docs))

    return all_docs

In [32]:
docs = load_all_pdfs()

Loaded: HARYANA.pdf → 75 pages
Loaded: UP.pdf → 218 pages

Total PDFs: 2
Total pages: 293


In [33]:
print(type(docs))
print(type(docs[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [35]:
print(docs[0].page_content[:500])

THE HARYANA MOTOR VEHICLES RULES, 1993  Notification The 30th July, 1993   G.S.R.44/C.A.59/88/S.28, 38, 65, 93, 95, 96, 107, 111 & 213/93.-- In exercise of the powers conferred by Sections 28, 38, 65, 93, 95, 96, 107, 111 and 213 of the MOTOR VEHICLES ACT, 1988 (Central Act of 1988) and all other powers enabling him in this behalf, the Governor of Haryana is hereby makes the following rules, namely :-  CHAPTER I PRELIMINARY  1. Short title and commencement.-- These rules may be called the Haryan


In [36]:
def combine_document_pages(docs):
    
    grouped_documents = {}

    for doc in docs:
        
        document_name = doc.metadata["document_name"]

        if document_name not in grouped_documents:
            grouped_documents[document_name] = []

        grouped_documents[document_name].append(doc)

    combined_documents = []

    for document_name, pages in grouped_documents.items():

        # Keep pages in original order
        pages = sorted(
            pages,
            key=lambda x: x.metadata.get("page", 0)
        )

        full_text = "\n".join(
            page.page_content for page in pages
        )

        metadata = pages[0].metadata.copy()

        metadata["document_name"] = document_name
        metadata["total_pages"] = len(pages)

        combined_documents.append(
            Document(
                page_content=full_text,
                metadata=metadata
            )
        )

    return combined_documents

In [37]:
combined_docs = combine_document_pages(docs)

print("Combined documents:", len(combined_docs))

Combined documents: 2


In [38]:
for doc in combined_docs:

    print("\nDocument:", doc.metadata["document_name"])
    print("Characters:", len(doc.page_content))


Document: HARYANA.pdf
Characters: 235970

Document: UP.pdf
Characters: 512542


In [40]:
print(combined_docs[1].page_content[:2000])

1 
 
THE UTTAR PRADESH MOTOR VEHICLE RULES, 19981  
(As Amended) 
In exercise of the powers under section 28, 38, 65, 95, 96, 107, 111, 138, 176  and 
213 of the Motor Vehicles A ct, 1998 (Act 59 of 1988) read with Section 21 of the 
General Clauses Act, 1987  (Act X of 1897) and in supersession of all existing of al l 
existing rules on the subject  the Governor is pleased to make the following rules after 
their previous publication, vide Government Notification No. 659 -T/30-4-67-89, dated 
March 21, 1995 published in th e Official Gazette of Uttar Pradesh, dated April 22, 1995 
as required under sub-section (1) of Section 212 of the said Act. 
CHAPTER I 
Preliminary 
1. Short Title— These R ules shall be called The Uttar Pradesh Motor 
Vehicles Rules, 1998. 
2. Definitions— In these Rules, unless here are many things repugnant in the 
subject or context— 
(i) "Act" means, the Motor Vehicles Act, 1988 (Act 59 of 1988); 
2[(i-a) "Accident" means an accident involving use of any m oto

In [41]:
def get_state(filename):

    filename = filename.lower()

    if "haryana" in filename:
        return "Haryana"

    if filename.startswith("up") or "uttar_pradesh" in filename:
        return "Uttar Pradesh"

    return "India"

In [42]:
print(get_state("HARYANA.pdf"))
print(get_state("UP.pdf"))

Haryana
Uttar Pradesh


In [43]:
def get_document_type(filename):

    filename = filename.lower()

    if "haryana" in filename:
        return "State Motor Vehicle Rules"

    if filename.startswith("up") or "uttar_pradesh" in filename:
        return "State Motor Vehicle Rules"

    if "cmvr" in filename or "central_motor" in filename:
        return "Central Motor Vehicle Rules"

    if "motor_vehicles_act" in filename or "motor_vehicle_act" in filename:
        return "Central Motor Vehicle Act"

    return "Motor Vehicle Law"

In [44]:
def split_into_rules(text):

    pattern = r'(?=\b\d+\.\s+[A-Z])'

    parts = re.split(pattern, text)

    return parts

In [45]:
haryana_text = combined_docs[0].page_content

rules = split_into_rules(haryana_text)

print("Detected parts:", len(rules))

Detected parts: 313


In [46]:
for rule in rules[:10]:

    print("=" * 80)
    print(rule[:500])

THE HARYANA MOTOR VEHICLES RULES, 1993  Notification The 30th July, 1993   G.S.R.44/C.A.59/88/S.28, 38, 65, 93, 95, 96, 107, 111 & 213/93.-- In exercise of the powers conferred by Sections 28, 38, 65, 93, 95, 96, 107, 111 and 213 of the MOTOR VEHICLES ACT, 1988 (Central Act of 1988) and all other powers enabling him in this behalf, the Governor of Haryana is hereby makes the following rules, namely :-  CHAPTER I PRELIMINARY  
1. Short title and commencement.-- These rules may be called the Haryana Motor Vehicles Rules, 1993.  
2. Definitions.--In these rules, unless there is anything repungnant to the subject or context:-  (a) "Act" means the Motor Vehicles Act, 1988 (Central Act 59 of 1988); (b) "Board of Inspection" means a Board of Inspection constituted under rule 37; (c) "Central Rules" means the Central Motor Vehicles Rules, 1989; (d) "Chapter" means a Chapter of these rules; (e) "Government" means the Government of the State of Haryana in the Administrative Department; (f) "Pass

In [47]:
def get_rule_number(text):

    match = re.match(
        r'^\s*(\d+)\.\s+',
        text
    )

    if match:
        return match.group(1)

    return None

In [48]:
print(get_rule_number("1. Short title and commencement"))
print(get_rule_number("9. Authority for making appointment"))

1
9


## Chunks

In [49]:
def create_legal_rule_chunks(combined_docs):

    legal_chunks = []

    for doc in combined_docs:

        text = doc.page_content
        filename = doc.metadata["document_name"]

        state = get_state(filename)
        document_type = get_document_type(filename)

        rule_parts = split_into_rules(text)

        for part in rule_parts:

            part = part.strip()

            if not part:
                continue

            rule_number = get_rule_number(part)

            metadata = {
                "document_name": filename,
                "state": state,
                "document_type": document_type,
                "rule": rule_number
            }

            legal_chunks.append(
                Document(
                    page_content=part,
                    metadata=metadata
                )
            )

    return legal_chunks

In [50]:
legal_chunks = create_legal_rule_chunks(combined_docs)

print("Total legal chunks:", len(legal_chunks))

Total legal chunks: 1224


In [52]:
print(legal_chunks[1].page_content)
print(legal_chunks[1].metadata)

1. Short title and commencement.-- These rules may be called the Haryana Motor Vehicles Rules, 1993.
{'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'rule': '1'}


In [53]:
for chunk in legal_chunks:

    if chunk.metadata["document_name"] == "UP.pdf":

        print(chunk.metadata)
        print(chunk.page_content[:500])

        break

{'document_name': 'UP.pdf', 'state': 'Uttar Pradesh', 'document_type': 'State Motor Vehicle Rules', 'rule': None}
1 
 
THE UTTAR PRADESH MOTOR VEHICLE RULES, 19981  
(As Amended) 
In exercise of the powers under section 28, 38, 65, 95, 96, 107, 111, 138, 176  and 
213 of the Motor Vehicles A ct, 1998 (Act 59 of 1988) read with Section 21 of the 
General Clauses Act, 1987  (Act X of 1897) and in supersession of all existing of al l 
existing rules on the subject  the Governor is pleased to make the following rules after 
their previous publication, vide Government Notification No. 659 -T/30-4-67-89, dated 
M


In [54]:
sub_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150
)

In [56]:
def create_final_chunks(legal_chunks):

    final_chunks = []

    for chunk in legal_chunks:

        text = chunk.page_content

        # Small enough → keep the complete legal rule
        if len(text) <= 1200:

            final_chunks.append(chunk)

        # Very large rule → split into smaller pieces
        else:

            smaller_chunks = sub_chunk_splitter.split_documents(
                [chunk]
            )

            for i, sub_chunk in enumerate(smaller_chunks):

                sub_chunk.metadata["sub_chunk"] = i + 1

                final_chunks.append(sub_chunk)

    return final_chunks

In [57]:
final_chunks = create_final_chunks(legal_chunks)

print("Final chunks:", len(final_chunks))

Final chunks: 1560


In [58]:
print(final_chunks[0].page_content)
print(final_chunks[0].metadata)

THE HARYANA MOTOR VEHICLES RULES, 1993  Notification The 30th July, 1993   G.S.R.44/C.A.59/88/S.28, 38, 65, 93, 95, 96, 107, 111 & 213/93.-- In exercise of the powers conferred by Sections 28, 38, 65, 93, 95, 96, 107, 111 and 213 of the MOTOR VEHICLES ACT, 1988 (Central Act of 1988) and all other powers enabling him in this behalf, the Governor of Haryana is hereby makes the following rules, namely :-  CHAPTER I PRELIMINARY
{'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'rule': None}


In [59]:
for chunk in final_chunks:

    if chunk.metadata.get("rule") == "138":

        print(chunk.metadata)
        print(chunk.page_content[:500])
        print("-" * 80)

{'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'rule': '138'}
138. Limit of seating capacity. [Section 111(2)(a)].-- (1) Not withstanding anything contained in these rules, no public service vehicle other than a motor club, shall be registered for a number of passengers in excess of the number obtained by subtracting 118 kilograms from the difference in kilograms between the registered laden and unladen weight of the vehicle and dividing the resulting figures by 160 in case of a single decked vehicle and 130 in the case of double decked vehicle or for such
--------------------------------------------------------------------------------


In [60]:
def combine_document_pages_with_page_map(docs):

    grouped_documents = {}

    for doc in docs:

        document_name = doc.metadata["document_name"]

        if document_name not in grouped_documents:
            grouped_documents[document_name] = []

        grouped_documents[document_name].append(doc)

    combined_documents = []

    for document_name, pages in grouped_documents.items():

        pages = sorted(
            pages,
            key=lambda x: x.metadata.get("page", 0)
        )

        full_text = ""
        page_map = []

        for page in pages:

            page_number = page.metadata.get("page", 0)
            page_label = page.metadata.get(
                "page_label",
                page_number + 1
            )

            start = len(full_text)

            full_text += page.page_content + "\n"

            end = len(full_text)

            page_map.append({
                "start": start,
                "end": end,
                "page": page_number,
                "page_label": page_label
            })

        metadata = pages[0].metadata.copy()

        metadata["document_name"] = document_name
        metadata["total_pages"] = len(pages)

        combined_documents.append(
            Document(
                page_content=full_text,
                metadata=metadata
            )
        )

        # page map alag attach karenge
        combined_documents[-1].metadata["page_map"] = page_map

    return combined_documents

In [61]:
combined_docs = combine_document_pages_with_page_map(docs)

print("Combined documents:", len(combined_docs))

Combined documents: 2


In [62]:
print(combined_docs[0].metadata["page_map"][:3])

[{'start': 0, 'end': 2748, 'page': 0, 'page_label': '1'}, {'start': 2748, 'end': 6481, 'page': 1, 'page_label': '2'}, {'start': 6481, 'end': 10068, 'page': 2, 'page_label': '3'}]


In [63]:
def get_chapter_at_position(text, position):

    chapter_pattern = r'CHAPTER\s+[IVXLCDM]+'

    chapters = list(
        re.finditer(
            chapter_pattern,
            text,
            re.IGNORECASE
        )
    )

    current_chapter = None

    for match in chapters:

        if match.start() <= position:
            current_chapter = match.group(0).strip()
        else:
            break

    return current_chapter

In [64]:
def get_pages_for_position(page_map, start_position, end_position):

    pages = []

    for page in page_map:

        if page["end"] > start_position and page["start"] < end_position:
            pages.append(page)

    if not pages:
        return None, None

    page_start = pages[0]["page_label"]
    page_end = pages[-1]["page_label"]

    return page_start, page_end

In [65]:
def create_legal_rule_chunks(combined_docs):

    legal_chunks = []

    for doc in combined_docs:

        text = doc.page_content
        filename = doc.metadata["document_name"]

        state = get_state(filename)
        document_type = get_document_type(filename)

        page_map = doc.metadata["page_map"]

        # Detect numbered rules
        pattern = r'(?=\b\d+\.\s+[A-Z])'

        parts = list(
            re.finditer(pattern, text)
        )

        for i, match in enumerate(parts):

            start = match.start()

            if i + 1 < len(parts):
                end = parts[i + 1].start()
            else:
                end = len(text)

            part = text[start:end].strip()

            if not part:
                continue

            rule_number = get_rule_number(part)

            chapter = get_chapter_at_position(
                text,
                start
            )

            page_start, page_end = get_pages_for_position(
                page_map,
                start,
                end
            )

            metadata = {
                "document_name": filename,
                "state": state,
                "document_type": document_type,
                "chapter": chapter,
                "rule": rule_number,
                "page_start": page_start,
                "page_end": page_end
            }

            legal_chunks.append(
                Document(
                    page_content=part,
                    metadata=metadata
                )
            )

    return legal_chunks

In [66]:
legal_chunks = create_legal_rule_chunks(combined_docs)

print("Total legal chunks:", len(legal_chunks))

Total legal chunks: 1222


In [67]:
for chunk in legal_chunks:

    if chunk.metadata.get("rule") == "138":

        print(chunk.metadata)
        print()
        print(chunk.page_content[:500])
        break

{'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'Chapter V', 'rule': '138', 'page_start': '49', 'page_end': '50'}

138. Limit of seating capacity. [Section 111(2)(a)].-- (1) Not withstanding anything contained in these rules, no public service vehicle other than a motor club, shall be registered for a number of passengers in excess of the number obtained by subtracting 118 kilograms from the difference in kilograms between the registered laden and unladen weight of the vehicle and dividing the resulting figures by 160 in case of a single decked vehicle and 130 in the case of double decked vehicle or for such


In [68]:
len(chunk.page_content)

904

In [69]:
for chunk in legal_chunks:

    if chunk.metadata.get("rule") == "138":
        print("Characters:", len(chunk.page_content))
        break

Characters: 904


In [70]:
sub_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150
)

In [71]:
def create_final_chunks(legal_chunks):

    final_chunks = []

    for chunk in legal_chunks:

        text = chunk.page_content

        # If legal rule is already small,
        # keep the complete rule.
        if len(text) <= 1200:

            chunk.metadata["sub_chunk"] = 1
            final_chunks.append(chunk)

        else:

            smaller_chunks = sub_chunk_splitter.split_documents(
                [chunk]
            )

            for i, sub_chunk in enumerate(smaller_chunks):

                # Preserve original legal metadata
                sub_chunk.metadata.update(
                    chunk.metadata
                )

                sub_chunk.metadata["sub_chunk"] = i + 1

                final_chunks.append(sub_chunk)

    return final_chunks

In [72]:
final_chunks = create_final_chunks(legal_chunks)

print("Final chunks:", len(final_chunks))

Final chunks: 1558


In [73]:
for chunk in final_chunks:

    if chunk.metadata.get("rule") == "138":

        print(chunk.metadata)
        print("Characters:", len(chunk.page_content))
        print(chunk.page_content[:300])
        print("-" * 80)

{'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'Chapter V', 'rule': '138', 'page_start': '49', 'page_end': '50', 'sub_chunk': 1}
Characters: 904
138. Limit of seating capacity. [Section 111(2)(a)].-- (1) Not withstanding anything contained in these rules, no public service vehicle other than a motor club, shall be registered for a number of passengers in excess of the number obtained by subtracting 118 kilograms from the difference in kilogr
--------------------------------------------------------------------------------


In [74]:
!pip install -U sentence-transformers faiss-cpu

  Using cached regex-2026.7.19-cp313-cp313-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 9.4 MB/s eta 0:00:000:00:0136m0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 9.1 MB/s eta 0:00:00
Using cached click-8.4.2-py3-none-any.whl (119 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 9.9 MB/s eta 0:00:0011.3 MB/s eta 0:00:01
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 11.4 MB/s eta 0:00:00 MB/s eta 0:00:01
Using cached regex-2026.7.19-cp313-cp313-macosx_11_0_arm64.whl (291 kB)
Using cached safetensors-0.8.0-cp310-abi3-mac

In [75]:
from sentence_transformers import SentenceTransformer

In [76]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [77]:
texts = [
    chunk.page_content
    for chunk in final_chunks
]

print("Total texts:", len(texts))

Total texts: 1558


In [78]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

In [79]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(1558, 384)


In [80]:
import faiss
import numpy as np

In [81]:
embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

In [82]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

In [83]:
index.add(embeddings)

print("Vectors stored:", index.ntotal)

Vectors stored: 1558


In [84]:
len(final_chunks)

1558

In [85]:
query = "Haryana mein seating capacity ka rule kya hai?"

query_embedding = embedding_model.encode(
    [query]
)

query_embedding = np.asarray(
    query_embedding,
    dtype="float32"
)

In [86]:
k = 5

distances, indices = index.search(
    query_embedding,
    k
)

In [87]:
for i, idx in enumerate(indices[0]):

    print("=" * 80)
    print("Rank:", i + 1)
    print("Distance:", distances[0][i])
    print("Metadata:", final_chunks[idx].metadata)
    print()
    print(final_chunks[idx].page_content[:500])

Rank: 1
Distance: 15.913103
Metadata: {'document_name': 'UP.pdf', 'state': 'Uttar Pradesh', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'CHAPTER X', 'rule': '65', 'page_start': '118', 'page_end': '118', 'sub_chunk': 1}

65. Sant Kabir Nagar UDQ
Rank: 2
Distance: 16.597897
Metadata: {'document_name': 'UP.pdf', 'state': 'Uttar Pradesh', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'CHAPTER X', 'rule': '48', 'page_start': '118', 'page_end': '118', 'sub_chunk': 1}

48. Lalitpur UCZ
Rank: 3
Distance: 16.827454
Metadata: {'document_name': 'UP.pdf', 'state': 'Uttar Pradesh', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'CHAPTER X', 'rule': '67', 'page_start': '118', 'page_end': '118', 'sub_chunk': 1}

67. Shahjahanpur UDS
Rank: 4
Distance: 16.935844
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'Chapter V', 'rule': '138', 'page_start': '49', 'page_end': '50', 'sub_chunk': 1}

138. Li

In [88]:
haryana_indices = [
    i
    for i, chunk in enumerate(final_chunks)
    if chunk.metadata.get("state") == "Haryana"
]

print("Haryana chunks:", len(haryana_indices))

Haryana chunks: 418


In [89]:
up_indices = [
    i
    for i, chunk in enumerate(final_chunks)
    if chunk.metadata.get("state") == "Uttar Pradesh"
]

print("UP chunks:", len(up_indices))

UP chunks: 1140


In [90]:
haryana_embeddings = np.asarray(
    [
        embeddings[i]
        for i in haryana_indices
    ],
    dtype="float32"
)

haryana_index = faiss.IndexFlatL2(
    haryana_embeddings.shape[1]
)

haryana_index.add(haryana_embeddings)

print("Haryana vectors:", haryana_index.ntotal)

Haryana vectors: 418


In [91]:
query = "Haryana mein seating capacity ka rule kya hai?"

query_embedding = embedding_model.encode(
    [query]
)

query_embedding = np.asarray(
    query_embedding,
    dtype="float32"
)

k = 5

distances, local_indices = haryana_index.search(
    query_embedding,
    k
)

In [92]:
for rank, local_idx in enumerate(local_indices[0]):

    original_idx = haryana_indices[local_idx]

    chunk = final_chunks[original_idx]

    print("=" * 80)
    print("Rank:", rank + 1)
    print("Distance:", distances[0][rank])
    print("Metadata:", chunk.metadata)
    print()
    print(chunk.page_content[:500])

Rank: 1
Distance: 16.935844
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'Chapter V', 'rule': '138', 'page_start': '49', 'page_end': '50', 'sub_chunk': 1}

138. Limit of seating capacity. [Section 111(2)(a)].-- (1) Not withstanding anything contained in these rules, no public service vehicle other than a motor club, shall be registered for a number of passengers in excess of the number obtained by subtracting 118 kilograms from the difference in kilograms between the registered laden and unladen weight of the vehicle and dividing the resulting figures by 160 in case of a single decked vehicle and 130 in the case of double decked vehicle or for such
Rank: 2
Distance: 18.164257
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'Chapter V', 'rule': '135', 'page_start': '48', 'page_end': '48', 'sub_chunk': 1}

135. Seating space. [Section 111(2)(a

In [93]:
original_idx = haryana_indices[local_idx]

In [94]:
!pip install rank-bm25


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [95]:
from rank_bm25 import BM25Okapi

In [96]:
haryana_texts = [
    final_chunks[i].page_content
    for i in haryana_indices
]


In [97]:
haryana_tokens = [
    text.lower().split()
    for text in haryana_texts
]


In [98]:
bm25 = BM25Okapi(haryana_tokens)

In [99]:
query = "Haryana mein seating capacity ka rule kya hai?"

query_tokens = query.lower().split()

bm25_scores = bm25.get_scores(query_tokens)

In [100]:
top_bm25 = np.argsort(bm25_scores)[::-1][:5]

In [101]:
for rank, local_idx in enumerate(top_bm25):

    original_idx = haryana_indices[local_idx]

    chunk = final_chunks[original_idx]

    print("=" * 80)
    print("Rank:", rank + 1)
    print("BM25 Score:", bm25_scores[local_idx])
    print("Metadata:", chunk.metadata)
    print()
    print(chunk.page_content[:500])

Rank: 1
BM25 Score: 11.470653248986963
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'CHAPTER V', 'rule': '63', 'page_start': '21', 'page_end': '21', 'sub_chunk': 1}

63. Limitation of capacity of stage carriages and contract carriages. [Section 96(2)(xv)].-- Save with the special permission of Government no permit or countersignature on the permit shall authorize the conveyance of more than fifty-four passengers, excluding the driver and the conductor in a stage carriage or contract carriage. Seats equal to 20% of the seating capacity shall be reserved for women.
Rank: 2
BM25 Score: 8.897499348567532
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'Chapter V', 'rule': '160', 'page_start': '54', 'page_end': '55', 'sub_chunk': 3}

be fitted with a bulb horn in addition to electric horn; and  (ix) be fitted with a rear view mirror mounted at a 

In [102]:
k = 20

vector_distances, vector_local_indices = haryana_index.search(
    query_embedding,
    k
)

In [103]:
bm25_top_k = 20

bm25_top_indices = np.argsort(
    bm25_scores
)[::-1][:bm25_top_k]

In [104]:
def reciprocal_rank_fusion(
    vector_indices,
    bm25_indices,
    k=60
):

    scores = {}

    # Vector rankings
    for rank, idx in enumerate(vector_indices, start=1):

        scores[idx] = scores.get(idx, 0) + (
            1 / (k + rank)
        )

    # BM25 rankings
    for rank, idx in enumerate(bm25_indices, start=1):

        scores[idx] = scores.get(idx, 0) + (
            1 / (k + rank)
        )

    # Sort by combined score
    ranked_results = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_results

In [105]:
hybrid_results = reciprocal_rank_fusion(
    vector_local_indices[0],
    bm25_top_indices
)

In [106]:
for rank, (local_idx, score) in enumerate(
    hybrid_results[:10],
    start=1
):

    original_idx = haryana_indices[local_idx]

    chunk = final_chunks[original_idx]

    print("=" * 80)
    print("Rank:", rank)
    print("RRF Score:", score)
    print("Metadata:", chunk.metadata)
    print()
    print(chunk.page_content[:400])

Rank: 1
RRF Score: 0.032266458495966696
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'CHAPTER V', 'rule': '63', 'page_start': '21', 'page_end': '21', 'sub_chunk': 1}

63. Limitation of capacity of stage carriages and contract carriages. [Section 96(2)(xv)].-- Save with the special permission of Government no permit or countersignature on the permit shall authorize the conveyance of more than fifty-four passengers, excluding the driver and the conductor in a stage carriage or contract carriage. Seats equal to 20% of the seating capacity shall be reserved for wom
Rank: 2
RRF Score: 0.02919863597612958
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'CHAPTER XII', 'rule': '7', 'page_start': '70', 'page_end': '70', 'sub_chunk': 1}

7. General Manager, Haryana Roadways.  Note :- In case of General Manager, Haryana Roadways the powers shall be exe

In [107]:
vector_scores = 1 / (1 + vector_distances[0])

vector_scores = (
    vector_scores - vector_scores.min()
) / (
    vector_scores.max() - vector_scores.min() + 1e-8
)

In [108]:
bm25_candidate_scores = bm25_scores[bm25_top_indices]

bm25_normalized = (
    bm25_candidate_scores - bm25_candidate_scores.min()
) / (
    bm25_candidate_scores.max() -
    bm25_candidate_scores.min() +
    1e-8
)

In [110]:
hybrid_scores = {}

for i, local_idx in enumerate(vector_local_indices[0]):

    hybrid_scores[local_idx] = (
        hybrid_scores.get(local_idx, 0)
        + 0.7 * vector_scores[i]
    )

In [111]:
for i, local_idx in enumerate(bm25_top_indices):

    hybrid_scores[local_idx] = (
        hybrid_scores.get(local_idx, 0)
        + 0.3 * bm25_normalized[i]
    )

In [112]:
hybrid_results = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

In [113]:
for rank, (local_idx, score) in enumerate(
    hybrid_results[:10],
    start=1
):

    original_idx = haryana_indices[local_idx]

    chunk = final_chunks[original_idx]

    print("=" * 80)
    print("Rank:", rank)
    print("Hybrid Score:", round(score, 4))
    print("Metadata:", chunk.metadata)
    print()
    print(chunk.page_content[:400])

Rank: 1
Hybrid Score: 0.7
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'Chapter V', 'rule': '138', 'page_start': '49', 'page_end': '50', 'sub_chunk': 1}

138. Limit of seating capacity. [Section 111(2)(a)].-- (1) Not withstanding anything contained in these rules, no public service vehicle other than a motor club, shall be registered for a number of passengers in excess of the number obtained by subtracting 118 kilograms from the difference in kilograms between the registered laden and unladen weight of the vehicle and dividing the resulting figure
Rank: 2
Hybrid Score: 0.6312
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'CHAPTER V', 'rule': '63', 'page_start': '21', 'page_end': '21', 'sub_chunk': 1}

63. Limitation of capacity of stage carriages and contract carriages. [Section 96(2)(xv)].-- Save with the special permission of Government

In [114]:
def hybrid_search(
    query,
    k_vector=20,
    k_bm25=20,
    top_k=5,
    vector_weight=0.7,
    bm25_weight=0.3
):

    # -------------------------
    # 1. Query embedding
    # -------------------------

    query_embedding = embedding_model.encode(
        [query]
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    # -------------------------
    # 2. Vector search
    # -------------------------

    vector_distances, vector_local_indices = (
        haryana_index.search(
            query_embedding,
            k_vector
        )
    )

    vector_distances = vector_distances[0]
    vector_local_indices = vector_local_indices[0]

    # Convert distance → similarity
    vector_scores = 1 / (1 + vector_distances)

    # Normalize
    vector_scores = (
        vector_scores - vector_scores.min()
    ) / (
        vector_scores.max()
        - vector_scores.min()
        + 1e-8
    )

    # -------------------------
    # 3. BM25 search
    # -------------------------

    query_tokens = query.lower().split()

    bm25_scores = bm25.get_scores(
        query_tokens
    )

    bm25_top_indices = np.argsort(
        bm25_scores
    )[::-1][:k_bm25]

    bm25_candidate_scores = (
        bm25_scores[bm25_top_indices]
    )

    # Normalize BM25
    bm25_normalized = (
        bm25_candidate_scores
        - bm25_candidate_scores.min()
    ) / (
        bm25_candidate_scores.max()
        - bm25_candidate_scores.min()
        + 1e-8
    )

    # -------------------------
    # 4. Hybrid scoring
    # -------------------------

    hybrid_scores = {}

    # Vector contribution
    for i, local_idx in enumerate(
        vector_local_indices
    ):

        hybrid_scores[local_idx] = (
            hybrid_scores.get(local_idx, 0)
            + vector_weight * vector_scores[i]
        )

    # BM25 contribution
    for i, local_idx in enumerate(
        bm25_top_indices
    ):

        hybrid_scores[local_idx] = (
            hybrid_scores.get(local_idx, 0)
            + bm25_weight * bm25_normalized[i]
        )

    # -------------------------
    # 5. Sort
    # -------------------------

    ranked_results = sorted(
        hybrid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # -------------------------
    # 6. Return top K
    # -------------------------

    results = []

    for local_idx, score in ranked_results[:top_k]:

        original_idx = haryana_indices[
            local_idx
        ]

        chunk = final_chunks[
            original_idx
        ]

        results.append({
            "score": float(score),
            "document": chunk
        })

    return results

In [115]:
results = hybrid_search(
    "Haryana mein seating capacity ka rule kya hai?"
)

In [116]:
for i, result in enumerate(results, start=1):

    print("=" * 80)
    print("Rank:", i)
    print("Score:", result["score"])
    print("Metadata:", result["document"].metadata)
    print()
    print(result["document"].page_content[:500])

Rank: 1
Score: 0.6999994516372681
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'Chapter V', 'rule': '138', 'page_start': '49', 'page_end': '50', 'sub_chunk': 1}

138. Limit of seating capacity. [Section 111(2)(a)].-- (1) Not withstanding anything contained in these rules, no public service vehicle other than a motor club, shall be registered for a number of passengers in excess of the number obtained by subtracting 118 kilograms from the difference in kilograms between the registered laden and unladen weight of the vehicle and dividing the resulting figures by 160 in case of a single decked vehicle and 130 in the case of double decked vehicle or for such
Rank: 2
Score: 0.6311966951861463
Metadata: {'document_name': 'HARYANA.pdf', 'state': 'Haryana', 'document_type': 'State Motor Vehicle Rules', 'chapter': 'CHAPTER V', 'rule': '63', 'page_start': '21', 'page_end': '21', 'sub_chunk': 1}

63. Limitation of capacity

In [117]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(results, start=1):

        doc = result["document"]

        metadata = doc.metadata

        source = (
            f"{metadata['document_name']}, "
            f"{metadata['state']}, "
            f"Rule {metadata['rule']}, "
            f"Pages {metadata['page_start']}-"
            f"{metadata['page_end']}"
        )

        context_parts.append(
            f"""
SOURCE {i}
{source}

{doc.page_content}
"""
        )

    return "\n".join(context_parts)

In [118]:
context = build_context(results)

print(context)


SOURCE 1
HARYANA.pdf, Haryana, Rule 138, Pages 49-50

138. Limit of seating capacity. [Section 111(2)(a)].-- (1) Not withstanding anything contained in these rules, no public service vehicle other than a motor club, shall be registered for a number of passengers in excess of the number obtained by subtracting 118 kilograms from the difference in kilograms between the registered laden and unladen weight of the vehicle and dividing the resulting figures by 160 in case of a single decked vehicle and 130 in the case of double decked vehicle or for such number of passenger that
when the vehicle is loaded in normal manner the axle weight of any axle will not exceed the registered axle weight for that axle.   (2) In addition to the number of persons permitted to be carried in a public service vehicle,-  (i) a child of not more than twelve years of age shall be reckoned as a half; and  (ii) a child of not more than three years of age shall be reckoned.


SOURCE 2
HARYANA.pdf, Haryana, Rule 63

In [119]:
!pip install ollama


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [120]:
import ollama

In [121]:
def create_prompt(question, context):

    prompt = f"""
You are NyayaDrive, an Indian Motor Vehicle Law Assistant.

Answer the user's question using ONLY the legal sources
provided below.

Rules:

1. Do not invent or assume legal provisions.
2. Do not use outside legal knowledge.
3. If the sources are insufficient, clearly say so.
4. Explain the answer in simple language.
5. Mention the relevant Rule/Section.
6. Always provide the source citation.
7. If multiple provisions are relevant, distinguish them clearly.

LEGAL SOURCES:

{context}

USER QUESTION:

{question}

ANSWER:
"""

    return prompt

In [122]:
question = "Haryana mein seating capacity ka rule kya hai?"

prompt = create_prompt(
    question,
    context
)

print(prompt)


You are NyayaDrive, an Indian Motor Vehicle Law Assistant.

Answer the user's question using ONLY the legal sources
provided below.

Rules:

1. Do not invent or assume legal provisions.
2. Do not use outside legal knowledge.
3. If the sources are insufficient, clearly say so.
4. Explain the answer in simple language.
5. Mention the relevant Rule/Section.
6. Always provide the source citation.
7. If multiple provisions are relevant, distinguish them clearly.

LEGAL SOURCES:


SOURCE 1
HARYANA.pdf, Haryana, Rule 138, Pages 49-50

138. Limit of seating capacity. [Section 111(2)(a)].-- (1) Not withstanding anything contained in these rules, no public service vehicle other than a motor club, shall be registered for a number of passengers in excess of the number obtained by subtracting 118 kilograms from the difference in kilograms between the registered laden and unladen weight of the vehicle and dividing the resulting figures by 160 in case of a single decked vehicle and 130 in the case o

In [123]:
def format_citation(metadata):
    return (
        f"{metadata['document_name']} | "
        f"{metadata['state']} | "
        f"Rule {metadata['rule']} | "
        f"Pages {metadata['page_start']}-{metadata['page_end']}"
    )

In [124]:
metadata = results[0]["document"].metadata

print(format_citation(metadata))

HARYANA.pdf | Haryana | Rule 138 | Pages 49-50


In [125]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(results, start=1):

        doc = result["document"]
        metadata = doc.metadata

        citation = format_citation(metadata)

        context_parts.append(
            f"""
SOURCE [{i}]
Citation: {citation}

Legal Text:
{doc.page_content}
"""
        )

    return "\n".join(context_parts)

In [126]:
def create_prompt(question, context):

    return f"""
You are NyayaDrive, an Indian Motor Vehicle Law Assistant.

Answer the user's question ONLY using the legal sources
provided below.

Rules:

1. Do not use outside legal knowledge.
2. Do not invent legal provisions.
3. If the sources are insufficient, say so clearly.
4. Explain the answer in simple language.
5. Mention the relevant Rule or Section.
6. Every important legal claim must have a source citation.
7. Use the SOURCE number provided in the context.
8. Never create or modify a citation.
9. If multiple sources are relevant, distinguish them.
10. If a source is only partially relevant, do not use it.

Citation format:

[Source 1]
[Source 2]

LEGAL SOURCES:

{context}

USER QUESTION:

{question}

ANSWER:
"""

In [ ]:
final_response = {
    "answer": llm_answer,
    "sources": [
        {
            "source_id": i + 1,
            "document": result["document"].metadata["document_name"],
            "state": result["document"].metadata["state"],
            "rule": result["document"].metadata.get("rule"),
            "chapter": result["document"].metadata.get("chapter"),
            "page_start": result["document"].metadata.get("page_start"),
            "page_end": result["document"].metadata.get("page_end")
        }
        for i, result in enumerate(results)
    ]
}